In [1]:
from skidl.logger import stop_log_file_output
stop_log_file_output(True)

In [2]:
from python.spice_tools import search_spice_model, save_part_model

entry = search_spice_model(name="FDMC510P-MS", library="Triode_MOS_Tube_Transistor")
if entry:
    print(entry["model_content"])
else:
    print("Not in model db")
    ## Save part model
    # save_part_model(
    # name="FDMC510P-MS",
    # library="Device",
    # model_content=".MODEL D1 D\n",
    # vendor_provided=True,
    # )


* FDMC510P-MS P-channel MOSFET (DFN3x3-8)
* Pin order: S1 S2 S3 G D
.SUBCKT FDMC510P_MS S1 S2 S3 G D
RS1 S1 S 0.2m
RS2 S2 S 0.2m
RS3 S3 S 0.2m
RDRAIN D D_INT 0.5m
CGS G S 3n
CGD G D_INT 1n
MMAIN D_INT S G S FDMC510P_PMOS L=1u W=2000u
.MODEL FDMC510P_PMOS PMOS (LEVEL=1 VTO=-1.8 KP=60 RD=0.002 RS=0.002)
.ENDS FDMC510P_MS



In [3]:
# from pathlib import Path
# from python.spice_tools import convert_skidl_module

# convert_skidl_module(
#     input_path=Path("test_cases/case_3A_charger/skidl/modules/battery_protection.py"),
#     subckt_name="Battery_Protection",
#     output_path=Path("test_cases/case_3A_charger/spice/modules/battery_protection_pyspice.py"),
# )

In [4]:
import json

test_bench_path = "test_cases/case_3A_charger/testbench/battery_protection_schema_valid.json"

with open(test_bench_path, "r") as f:
    test_bench = json.load(f)
    testcases = test_bench["use_cases"]
case_ids = [case["name"] for case in testcases if case["mode"] == "auto"]

case_ids

['startup_reverse_block_with_charger_ramp',
 'forward_conduction_drop_3A_25C',
 'forward_conduction_drop_3A_60C',
 'reverse_blocking_usb_removed',
 'overcharge_overvoltage_cutoff',
 'overdischarge_cutoff_under_load',
 'steady_ripple_pass_through']

In [5]:
from python.spice_tools.testbench_runner import run_use_case

idx = 3
harness_path = f"test_harness_unit/{case_ids[idx]}.py"

reports = run_use_case(
    schema_path=test_bench_path,
    harness_path=harness_path,
    use_case_name=case_ids[idx],
)
for report in reports:
    print(report)           


MeasurementResult(use_case='reverse_blocking_usb_removed', measurement_id='reverse_leakage_usb_removed', value=4.0018646744551525e-08, op='<=', limit=0.001, passed=True)
MeasurementResult(use_case='reverse_blocking_usb_removed', measurement_id='pack_voltage_hold', value=1.0, op='>=', limit=0.95, passed=True)


In [6]:
reports

[MeasurementResult(use_case='reverse_blocking_usb_removed', measurement_id='reverse_leakage_usb_removed', value=4.0018646744551525e-08, op='<=', limit=0.001, passed=True),
 MeasurementResult(use_case='reverse_blocking_usb_removed', measurement_id='pack_voltage_hold', value=1.0, op='>=', limit=0.95, passed=True)]